# Multiclass SBM for Rotating-Machine Fault Diagnosis - Step-by-Step Experiment Reproduction

This notebook guides you step-by-step through reproducing the core experiments and configurations presented in the research paper:
**["Improved similarity-based modeling for the classification of rotating-machine failures"](https://doi.org/10.1016/j.jfranklin.2017.07.038)** (Journal of the Franklin Institute, 2018).

We leverage our modular, strongly typed, and optimized Python scripts to run feature extraction, build clean SBM class dictionaries, perform state estimations, and evaluate classification ensembles.

## Setup Environment and Imports

First, we make sure that the parent directory containing the modular pipeline scripts is appended to our system path. This allows us to load `data_prep`, `feature_extraction`, `sbm_model`, `rf_classifier`, and `cv_tuning` cleanly.

In [1]:
import os
import sys
import pickle

# Append parent directory to sys.path to load codebase libraries
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

try:
    from mafaulda import data_prep
    from mafaulda import feature_extraction
    from mafaulda import sbm_model
    from mafaulda import rf_classifier
    from mafaulda import cv_tuning
    print("\u2713 All modular SBM pipeline components successfully loaded!")
except ImportError as e:
    print(f"\u2717 Failed to import pipeline scripts: {e}")

✓ All modular SBM pipeline components successfully loaded!


## Step 1: Data Mapping & Stratified Train/Test Split

We map all CSV operational scenario files within the **Machinery Fault Database (MaFaulDa)** and assign them labels based on directory structure. 
Then, we perform a deterministic stratified split (89% training, 11% testing) that matches the paper's original setup, guaranteeing that the scarce **Normal** class is proportionally represented in both splits.

In [2]:
# Configure dataset directory path (resolve home directory)
dataset_path = os.path.expanduser('~/datasets/mafaulda')
print(f"Target MaFaulDa directory: {dataset_path}")

if os.path.exists(dataset_path):
    train_paths, test_paths, train_labels, test_labels = data_prep.map_dataset(dataset_path)
    print(f"\n\u2713 Successfully mapped {len(train_paths) + len(test_paths)} scenarios:")
    print(f"  Training scenarios count: {len(train_paths)}")
    print(f"  Testing scenarios count:  {len(test_paths)}")
else:
    print(f"\u26a0\ufe0f Raw dataset not found at '{dataset_path}'.")
    print("We will fall back to using pre-extracted features in the subsequent steps if they exist in the cache.")

Target MaFaulDa directory: /home/tito/datasets/mafaulda

✓ Successfully mapped 1951 scenarios:
  Training scenarios count: 1759
  Testing scenarios count:  192


## Step 2: Signal Feature Extraction

For each time-series scenario file (each composed of 8 sensor channels sampled at 50 kHz for 5 seconds), we compress the signals into a 46-dimensional hand-crafted diagnostic vector:
* **1 Rotation Frequency**: Peak estimation from tachometer's Discrete Fourier Transform (DFT).
* **21 Spectral Magnitudes**: Continuous peak interpolation for first 7 physical sensors at the fundamental rotation speed, second harmonic, and third harmonic.
* **24 Statistical Descriptors**: Mean, 100-bin Shannon Entropy, and Fisher Kurtosis calculated for all 8 channels.

We can toggle our advanced Digital Signal Processing (DSP) enhancements:
* `use_hann`: Applies Hanning window with exact coherent gain correction to eliminate spectral leakage.
* `use_fixed_entropy`: Locks histogram range strictly to `(-10.0, 10.0)` to prevent dynamic scale distortion from impulsive shock spikes.

In [3]:
# Toggle DSP features
use_hann = True
use_fixed_entropy = True

if os.path.exists(dataset_path):
    # Extract features concurrently using multi-core process workers
    X_train = feature_extraction.process_set_parallel(train_paths, "Training", use_hann, use_fixed_entropy)
    X_test = feature_extraction.process_set_parallel(test_paths, "Testing", use_hann, use_fixed_entropy)

    # Cache the matrices inside the data/ folder
    os.makedirs('../data', exist_ok=True)
    with open('../data/X_train_features.pkl', 'wb') as f:
        pickle.dump(X_train, f)
    with open('../data/X_test_features.pkl', 'wb') as f:
        pickle.dump(X_test, f)
    with open('../data/y_train.pkl', 'wb') as f:
        pickle.dump(train_labels, f)
    with open('../data/y_test.pkl', 'wb') as f:
        pickle.dump(test_labels, f)
    print("\n\u2713 Successfully extracted and cached feature matrices.")
else:
    # Try to load pre-extracted matrices from cache
    try:
        with open('../data/X_train_features.pkl', 'rb') as f:
            X_train = pickle.load(f)
        with open('../data/X_test_features.pkl', 'rb') as f:
            X_test = pickle.load(f)
        with open('../data/y_train.pkl', 'rb') as f:
            train_labels = pickle.load(f)
        with open('../data/y_test.pkl', 'rb') as f:
            test_labels = pickle.load(f)
        print("\u2713 Loaded pre-extracted feature matrices from cache:")
        print(f"  X_train: {len(X_train)} samples")
        print(f"  X_test:  {len(X_test)} samples")
    except FileNotFoundError:
        print("\u2717 Cache files not found! Please place the MaFaulDa raw dataset in '~/datasets/mafaulda'.")


Starting feature extraction for Training set (1759 samples) in parallel...

Starting feature extraction for Testing set (192 samples) in parallel...

✓ Successfully extracted and cached feature matrices.


## Step 3: SBM Model B (92-Dimensional Error Residuals)

Models the normal operational manifold of each of the 6 fault classes:
1. **Geometric Median Initialization**: Employs Weiszfeld's algorithm for an iterative approximation of the class centroid, providing robust anchoring against signal outliers.
2. **Memory Matrix Construction (Threshold Method)**: Iteratively grows a dictionary matrix $D(c)$ using the Wegerich Similarity Function (WSF) and $L_1$ norm. A candidate state is memorized only if its similarity to all existing states is strictly less than threshold $\tau = 0.85$.
3. **State Estimation**: Reconstructs a clean estimate vector $\hat{x}(n, c)$ using Moore-Penrose pseudo-inversion of the symmetric pairwise similarity matrix $G(c)$ and $L_1$ weight normalization.
4. **Feature Extension**: Appends the 46-dimensional best-estimate SBM residual error vector $e(n) = x(n) - \hat{x}(n, c^*)$ to the original features, producing a 92-dimensional representation: $x_{\text{ext}}(n) = [x(n)^T, e(n)^T]^T$.

In [4]:
unique_classes = sorted(list(set(train_labels)))
D_c_dict = {}

print("Constructing SBM Class State Dictionaries:")
for c in unique_classes:
    X_c = [X_train[i] for i, lbl in enumerate(train_labels) if lbl == c]

    # Seed with Weiszfeld's median and build dictionary via threshold method
    D_c = sbm_model.construct_class_dictionary(X_c, tau=0.85, gamma=0.0010)
    D_c_dict[c] = D_c
    print(f"  \u2022 Class '{c}': Dictionary has {len(D_c)} representative states (from {len(X_c)} samples).")

# Generate 92-dimensional extended features for training and testing
print("\nGenerating SBM Model B 92-dimensional extended error residual features...")
X_train_ext = sbm_model.generate_extended_features(X_train, D_c_dict, gamma=0.0010)
X_test_ext = sbm_model.generate_extended_features(X_test, D_c_dict, gamma=0.0010)

print(f"\n\u2713 SBM Model B features generated successfully:")
print(f"  X_train_ext size: {len(X_train_ext)}")
print(f"  X_test_ext size:  {len(X_test_ext)}")

Constructing SBM Class State Dictionaries:
  • Class 'horizontal-misalignment': Dictionary has 1 representative states (from 178 samples).
  • Class 'imbalance': Dictionary has 39 representative states (from 300 samples).
  • Class 'normal': Dictionary has 1 representative states (from 45 samples).
  • Class 'overhang': Dictionary has 1 representative states (from 462 samples).
  • Class 'underhang': Dictionary has 32 representative states (from 503 samples).
  • Class 'vertical-misalignment': Dictionary has 1 representative states (from 271 samples).

Generating SBM Model B 92-dimensional extended error residual features...

✓ SBM Model B features generated successfully:
  X_train_ext size: 1759
  X_test_ext size:  192


### Train and Evaluate SBM Model B Ensemble Classifier

We train the Random Forest ensemble on the 92 extended features. The model leverages balanced class weights to resolve the natural data scarcity of the Normal operational class, ensuring flawless detection.

In [5]:
print("Training SBM Model B classifier...")
clf = rf_classifier.train_classifier(X_train_ext, train_labels)

print("\nEvaluating SBM Model B on Test Set:")
rf_classifier.evaluate_classifier(clf, X_test_ext, test_labels)

Training SBM Model B classifier...

Evaluating SBM Model B on Test Set:

================ EVALUATION RESULTS ================
Overall Classification Accuracy: 98.96%
Expected Accuracy from Paper:   ~98.49%

--- CONFUSION MATRIX (True \ Predicted) ---
True Class              | horizontal-misalignment |        imbalance        |         normal          |        overhang         |        underhang        |  vertical-misalignment  |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
horizontal-misalignment |           19            |            0            |            0            |            0            |            0            |            0            |
imbalance               |            0            |           33            |            0            |            0            |            0            |            0            |
normal               

['underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'overhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'vertical-misalignment',
 'vertical-misalignment',
 'vertical-misalignment',
 'vertical-misalignment',
 'vertical-misalignment',
 'horizontal-misalignment',
 'vertical-misalignment',
 'vertical-misalignment',
 'vertical-misalignme

## Step 4: SBM Experiment 3 Configuration 3 (52-Dimensional Similarities)

In this configuration, instead of using physical estimation error residuals, we feed SBM class similarities directly to the classifier.

For each sample $x(n)$, we compute the WSF similarity to its SBM estimate across all 6 classes, yielding a 6-dimensional similarity vector which is concatenated with the original 46 features to create a **52-dimensional extended feature matrix**:
$$x_{\text{ext}}(n) = \begin{bmatrix} x(n) \\ s(x(n), \hat{x}(n, c_1)) \\ \vdots \\ s(x(n), \hat{x}(n, c_6)) \end{bmatrix}$$

In [6]:
print("Generating SBM Experiment 3 Configuration 3 similarity-extended features...")
X_train_sim = sbm_model.generate_similarity_extended_features(X_train, D_c_dict, gamma=0.0010)
X_test_sim = sbm_model.generate_similarity_extended_features(X_test, D_c_dict, gamma=0.0010)

print(f"\n\u2713 Similarity-extended features generated successfully:")
print(f"  X_train_sim size: {len(X_train_sim)}")
print(f"  X_test_sim size:  {len(X_test_sim)}")

print("\nTraining Random Forest classifier on 52 similarity features...")
clf_sim = rf_classifier.train_classifier(X_train_sim, train_labels)

print("\nEvaluating SBM Similarity model on Test Set:")
rf_classifier.evaluate_classifier(clf_sim, X_test_sim, test_labels)

Generating SBM Experiment 3 Configuration 3 similarity-extended features...

✓ Similarity-extended features generated successfully:
  X_train_sim size: 1759
  X_test_sim size:  192

Training Random Forest classifier on 52 similarity features...

Evaluating SBM Similarity model on Test Set:

================ EVALUATION RESULTS ================
Overall Classification Accuracy: 98.96%
Expected Accuracy from Paper:   ~98.49%

--- CONFUSION MATRIX (True \ Predicted) ---
True Class              | horizontal-misalignment |        imbalance        |         normal          |        overhang         |        underhang        |  vertical-misalignment  |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
horizontal-misalignment |           19            |            0            |            0            |            0            |            0            |           

['underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'underhang',
 'vertical-misalignment',
 'vertical-misalignment',
 'vertical-misalignment',
 'vertical-misalignment',
 'vertical-misalignment',
 'horizontal-misalignment',
 'vertical-misalignment',
 'vertical-misalignment',
 'vertical-misalignm

## Step 5: SBM Hyperparameter Cross-Validation Tuning

To guarantee SBM model parameter optimization without data leakage, the codebase includes a Stratified 10-fold Cross-Validation grid search module. This runs a search over SBM sensitivity $\gamma$ and dictionary memorization threshold $\tau$ strictly on the training set.

In [7]:
print("Executing 10-Fold Stratified Cross-Validation Parameter Grid Search:")

# Define SBM hyperparameter values directly inside this cell
gamma_values = [0.0001, 0.0005, 0.0010, 0.0100, 0.1000]
tau_values = [0.75, 0.80, 0.85, 0.90]

print(f"Evaluating over Custom Grid:")
print(f"  \u2022 Gamma values: {gamma_values}")
print(f"  \u2022 Tau values:   {tau_values}")

# Execute the custom SBM tuning run
try:
    cv_tuning.run_tuning('../data', gammas=gamma_values, taus=tau_values)
except Exception as e:
    print(f"Error during cross-validation: {e}")

Executing 10-Fold Stratified Cross-Validation Parameter Grid Search:
Evaluating over Custom Grid:
  • Gamma values: [0.0001, 0.0005, 0.001, 0.01, 0.1]
  • Tau values:   [0.75, 0.8, 0.85, 0.9]

=== STEP 5: SBM Hyperparameter Tuning (10-Fold Stratified CV) ===
Loading training features from ../data...
  Training features samples: 1759
  Training labels samples:   1759

Setting up 10-fold Stratified Cross-Validation (random_state=42)...
Pre-computing geometric medians for all folds and classes...
Pre-computation of geometric medians finished in 1.22s.

Evaluating combination: gamma=0.0001, tau=0.75 -> Mean 10-Fold CV Accuracy: 96.59% (computed in 121.02s)
Evaluating combination: gamma=0.0001, tau=0.8 -> Mean 10-Fold CV Accuracy: 96.59% (computed in 121.44s)
Evaluating combination: gamma=0.0001, tau=0.85 -> Mean 10-Fold CV Accuracy: 96.48% (computed in 121.92s)
Evaluating combination: gamma=0.0001, tau=0.9 -> Mean 10-Fold CV Accuracy: 96.53% (computed in 121.55s)
Evaluating combination: ga